# Lọc 7 loại metrics từ Telegraf → Kafka

# What

In [ ]:
import json
import psycopg2
from confluent_kafka import Consumer
from datetime import datetime, timezone

In [55]:
KAFKA_BOOTSTRAP = "123.30.48.173:9092"
TOPIC = "metrics-raw"
GROUP_ID = "metrics-consumer-group"

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "metrics_db",
    "user": "mlops",
    "password": "mlops123",
}

In [72]:
consumer = Consumer({
    "bootstrap.servers": KAFKA_BOOTSTRAP,
    "group.id": GROUP_ID,
    "auto.offset.reset": "earliest",
})
consumer.subscribe([TOPIC])

msg = None
while (msg == None):
    msg = consumer.poll(1.0)
    continue

print(msg)

In [74]:
print(msg.value().decode('utf-8'))
raw_value: str = msg.value().decode('utf-8')

{"fields":{"load1":0.13,"load15":0.15,"load5":0.13,"n_cpus":4,"n_physical_cpus":4},"name":"system","tags":{"host":"intern-metrics-server-1"},"timestamp":1785480530}



## Trying to parse the message

In [75]:
ALLOWED_METRICS = {
    "cpu": {"usage_idle"},              # sẽ tự tính usage_active = 100 - usage_idle
    "mem": {"used_percent"},
    "disk": {"used_percent"},
    "net": {"bytes_recv", "bytes_sent", "err_in", "err_out"},
}

In [62]:
def get_db_conn():
    return psycopg2.connect(**DB_CONFIG)

In [65]:
def parse_telegraf_message(raw_value: str) -> list[dict]:
    data = json.loads(raw_value)
    metric_group = data.get("name", "unknown")

    # Lọc ngay từ đầu: bỏ qua toàn bộ group không nằm trong danh sách cần
    if metric_group not in ALLOWED_METRICS:
        return []

    host_id = data.get("tags", {}).get("host", "unknown-host")
    ts_unix = data.get("timestamp")
    ts = datetime.fromtimestamp(ts_unix, tz=timezone.utc) if ts_unix else datetime.now(timezone.utc)

    rows = []
    allowed_fields = ALLOWED_METRICS[metric_group]
    for field_name, value in data.get("fields", {}).items():
        if field_name not in allowed_fields:
            continue
        if not isinstance(value, (int, float)):
            continue

        # đặc biệt: đổi usage_idle thành usage_active cho dễ hiểu (CPU đang bận, không phải rảnh)
        if metric_group == "cpu" and field_name == "usage_idle":
            field_name = "usage_active"
            value = 100 - value

        metric_name = f"{metric_group}_{field_name}"
        rows.append({
            "ts": ts,
            "host_id": host_id,
            "metric_name": metric_name,
            "value": float(value),
        })
    return rows

In [76]:
rows = parse_telegraf_message(raw_value=raw_value)
rows

[]